<a href="https://colab.research.google.com/github/buddanaveenkumar-dev/gpu-credit/blob/main/gpu_gig_credit_decisioning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout)

Mon Jun 29 12:56:40 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   32C    P0             46W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

# GPU-Accelerated Credit Decisioning for Gig Economy Platforms

This notebook is a POC to explore how an end-to-end credit decisioning pipeline maps onto the NVIDIA accelerated computing stack.

The data is synthetic. The architecture patterns are based on real production credit systems.

The goal is simple to show how a credit decision can run on GPU from raw platform signals to fraud screening, risk scoring, limit calculation, explanation, and audit trail.

Stack used in this notebook:

- RAPIDS cuDF for GPU feature engineering
- GPU XGBoost for credit scoring
- NetworkX for the graph example (cuGraph in production)
- NVIDIA NIM style inference flow
- SHAP for explainability

In [2]:
# Install RAPIDS stack
!pip install \
    --extra-index-url=https://pypi.nvidia.com \
    cudf-cu12==25.4.* \
    cuml-cu12==25.4.* \
    cugraph-cu12==25.4.* \
    -q

!pip install xgboost shap openai -q

import cudf
import cuml
import cugraph
print(f"✓ cuDF version: {cudf.__version__}")
print(f"✓ cuML version: {cuml.__version__}")
print(f"✓ cuGraph version: {cugraph.__version__}")
print("✓ Full RAPIDS stack ready on A100")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 165.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 262.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 565.0/565.0 MB 62.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.6/27.6 MB 104.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9/46.9 MB 195.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 253.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 711.1/711.1 kB 245.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.9/135.9 kB 148.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.7/50.7 kB 72.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 424.6/424.6 MB 9.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 852.3/852.3 kB 229.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Stage 1 – Synthetic Data Generation

I generate a synthetic dataset representing gig workers on a delivery and rideshare platform.

The goal is to understand how a complete credit decision can run on GPU, starting with raw platform signals and ending with fraud screening, risk scoring, limit calculation, explainability, and audit logging.

Each worker contains information across four categories:

- Platform activity
- Earnings and payouts
- Identity signals
- Existing repayment obligations

Approximately 3% of records are marked as fraudulent by injecting synthetic identity patterns such as frequent device changes and unusually fast onboarding.

No PII is used.

In [3]:
import numpy as np
import pandas as pd

np.random.seed(42)
N = 5_000_000

print(f"Generating {N:,} synthetic gig worker records...")
print("Creating synthetic platform, payment, identity, and repayment signals...")

df = pd.DataFrame({
    # Platform earnings signals
    "monthly_earnings_usd":      np.random.lognormal(mean=7.8, sigma=0.4, size=N).round(2),
    "trips_per_month":           np.random.poisson(lam=180, size=N),
    "platform_tenure_months":    np.random.randint(1, 60, size=N),
    "fulfillment_rating":        np.clip(np.random.normal(4.5, 0.3, N), 1.0, 5.0).round(2),
    "active_days_per_month":     np.random.randint(10, 28, size=N),

    # Payment rail signals
    "wallet_txn_count_30d":      np.random.poisson(lam=45, size=N),
    "avg_disbursement_usd":      np.random.lognormal(mean=5.2, sigma=0.5, size=N).round(2),
    "payment_consistency_score": np.clip(np.random.normal(0.78, 0.12, N), 0, 1).round(3),

    # Identity signals
    "device_change_count_90d":   np.random.poisson(lam=0.4, size=N),
    "onboarding_velocity_days":  np.random.randint(1, 30, size=N),
    "kyc_verification_score":    np.clip(np.random.normal(0.85, 0.1, N), 0, 1).round(3),

    # Obligations
    "existing_loan_count":       np.random.poisson(lam=0.8, size=N),
    "foir_current":              np.clip(np.random.normal(0.38, 0.12, N), 0.05, 0.85).round(3),

    # Target
    "repaid_on_time":            np.random.binomial(1, p=0.78, size=N),
})

# Inject fraud patterns in 3% of records
fraud_idx = np.random.choice(N, size=int(N * 0.03), replace=False)
df.loc[fraud_idx, "device_change_count_90d"] += np.random.randint(3, 8, size=len(fraud_idx))
df.loc[fraud_idx, "onboarding_velocity_days"] = np.random.randint(1, 3, size=len(fraud_idx))
df.loc[fraud_idx, "repaid_on_time"] = 0
df["is_fraud"] = 0
df.loc[fraud_idx, "is_fraud"] = 1

print(f"✓ Dataset shape: {df.shape}")
print(f"✓ Repayment rate: {df.repaid_on_time.mean():.1%}")
print(f"✓ Fraud rate: {df.is_fraud.mean():.1%}")
df.head(3)

Generating 5,000,000 synthetic gig worker records...
Creating synthetic platform, payment, identity, and repayment signals...
✓ Dataset shape: (5000000, 15)
✓ Repayment rate: 75.7%
✓ Fraud rate: 3.0%


,monthly_earnings_usd,trips_per_month,platform_tenure_months,fulfillment_rating,active_days_per_month,wallet_txn_count_30d,avg_disbursement_usd,payment_consistency_score,device_change_count_90d,onboarding_velocity_days,kyc_verification_score,existing_loan_count,foir_current,repaid_on_time,is_fraud
0,2977.04,200,44,4.19,14,58,62.58,0.791,0,4,0.930,1,0.456,1,0
1,2309.29,195,10,4.79,24,47,208.73,1.000,1,25,0.969,2,0.179,1,0
2,3162.36,185,5,4.37,21,47,180.46,0.516,0,5,0.939,0,0.427,1,0


## Stage 2 – RAPIDS GPU Feature Engineering

For convenience, this section recreates the synthetic dataset from Stage 1 so the feature engineering benchmark can be run independently.

Feature engineering is usually one of the first bottlenecks in a credit decisioning pipeline as data volume grows.

This section attempts to use RAPIDS cuDF for GPU dataframe processing.

The notebook creates five feature groups:

1. Earnings velocity
2. Platform reliability
3. Repayment capacity
4. Identity risk
5. Composite credit signal

In [14]:
import numpy as np
import pandas as pd

np.random.seed(42)
N = 5_000_000

print(f"Generating {N:,} synthetic gig worker records...")

df = pd.DataFrame({
    "monthly_earnings_usd":      np.random.lognormal(mean=7.8, sigma=0.4, size=N).round(2),
    "trips_per_month":           np.random.poisson(lam=180, size=N),
    "platform_tenure_months":    np.random.randint(1, 60, size=N),
    "fulfillment_rating":        np.clip(np.random.normal(4.5, 0.3, N), 1.0, 5.0).round(2),
    "active_days_per_month":     np.random.randint(10, 28, size=N),
    "wallet_txn_count_30d":      np.random.poisson(lam=45, size=N),
    "avg_disbursement_usd":      np.random.lognormal(mean=5.2, sigma=0.5, size=N).round(2),
    "payment_consistency_score": np.clip(np.random.normal(0.78, 0.12, N), 0, 1).round(3),
    "device_change_count_90d":   np.random.poisson(lam=0.4, size=N),
    "onboarding_velocity_days":  np.random.randint(1, 30, size=N),
    "kyc_verification_score":    np.clip(np.random.normal(0.85, 0.1, N), 0, 1).round(3),
    "existing_loan_count":       np.random.poisson(lam=0.8, size=N),
    "foir_current":              np.clip(np.random.normal(0.38, 0.12, N), 0.05, 0.85).round(3),
    "repaid_on_time":            np.random.binomial(1, p=0.78, size=N),
})

fraud_idx = np.random.choice(N, size=int(N * 0.03), replace=False)
df.loc[fraud_idx, "device_change_count_90d"] += np.random.randint(3, 8, size=len(fraud_idx))
df.loc[fraud_idx, "onboarding_velocity_days"] = np.random.randint(1, 3, size=len(fraud_idx))
df.loc[fraud_idx, "repaid_on_time"] = 0
df["is_fraud"] = 0
df.loc[fraud_idx, "is_fraud"] = 1

print(f"✓ Dataset shape: {df.shape}")
print(f"✓ Repayment rate: {df.repaid_on_time.mean():.1%}")
print(f"✓ Fraud rate: {df.is_fraud.mean():.1%}")

Generating 5,000,000 synthetic gig worker records...
✓ Dataset shape: (5000000, 15)
✓ Repayment rate: 75.7%
✓ Fraud rate: 3.0%


In [16]:
import cudf
import time
import pandas as pd
import numpy as np

print("GPU vs CPU: Feature Engineering Benchmark")
print("=" * 55)

# Reuse the dataset created in Stage 1
print(f"Benchmarking feature engineering on {len(df):,} records...")

# CPU timing
t_cpu = time.time()
df_b = df.copy()
df_b["earnings_per_trip"]    = df_b["monthly_earnings_usd"] / (df_b["trips_per_month"] + 1)
df_b["earnings_stability"]   = (df_b["payment_consistency_score"] * 0.6 + (df_b["active_days_per_month"] / 28.0) * 0.4)
df_b["tenure_composite"]     = ((df_b["platform_tenure_months"] / 60.0) * 0.5 + (df_b["fulfillment_rating"] / 5.0) * 0.5)
df_b["repayment_capacity"]   = (1.0 - df_b["foir_current"]) * df_b["earnings_stability"]
df_b["composite_signal"]     = df_b["repayment_capacity"] * 0.35 + df_b["tenure_composite"] * 0.25
cpu_ms = (time.time() - t_cpu) * 1000

# GPU timing
t_gpu = time.time()
gdf_b = cudf.from_pandas(df)
gdf_b["earnings_per_trip"]   = gdf_b["monthly_earnings_usd"] / (gdf_b["trips_per_month"] + 1)
gdf_b["earnings_stability"]  = (gdf_b["payment_consistency_score"] * 0.6 + (gdf_b["active_days_per_month"] / 28.0) * 0.4)
gdf_b["tenure_composite"]    = ((gdf_b["platform_tenure_months"] / 60.0) * 0.5 + (gdf_b["fulfillment_rating"] / 5.0) * 0.5)
gdf_b["repayment_capacity"]  = (1.0 - gdf_b["foir_current"]) * gdf_b["earnings_stability"]
gdf_b["composite_signal"]    = gdf_b["repayment_capacity"] * 0.35 + gdf_b["tenure_composite"] * 0.25
gpu_ms = (time.time() - t_gpu) * 1000

speedup = cpu_ms / gpu_ms
print(f"  Records         : {len(df):,}")
print(f"  CPU, pandas     : {cpu_ms:>8.0f} ms")
print(f"  GPU, cuDF A100  : {gpu_ms:>8.0f} ms")
print(f"  Speedup         : {speedup:>8.1f}x")
print()
print("Context:")
print("  Tested on Google Colab A100-SXM4-40GB.")
print("  The benchmark includes transferring data from pandas to cuDF.")
print(f"  Even with that overhead, the GPU was {speedup:.1f}x faster for this workload.")
print("  Larger end-to-end pipelines benefit more when data remains on the GPU")
print("  across feature engineering, graph analytics, model training, and inference.")

GPU vs CPU: Feature Engineering Benchmark
Benchmarking feature engineering on 5,000,000 records...
  Records         : 5,000,000
  CPU, pandas     :      635 ms
  GPU, cuDF A100  :      388 ms
  Speedup         :      1.6x

Context:
  Tested on Google Colab A100-SXM4-40GB.
  The benchmark includes transferring data from pandas to cuDF.
  Even with that overhead, the GPU was 1.6x faster for this workload.
  Larger end-to-end pipelines benefit more when data remains on the GPU
  across feature engineering, graph analytics, model training, and inference.


Stage 3 – Graph-Based Fraud Detection

Fraud is not always visible from one worker record.

A worker can look clean in a tabular model, but the risk starts showing up when we look at relationships across workers, devices, bank accounts, and merchants.

In gig platforms, some fraud patterns come from shared devices, fast onboarding, repeated bank accounts, or small groups of accounts connected to the same merchant network.

This is why I added a graph fraud layer before credit scoring.

In this stage, I use cuGraph to build a simple relationship graph and identify suspicious connected communities. The goal is to show how graph-based fraud screening can catch patterns that row-level features may miss.

The fraud gate runs before credit evaluation. If a worker is flagged as high risk, the record does not move to the scoring layer.


In [5]:
import networkx as nx
import numpy as np
import time

print("=" * 55)
print("Stage 3: Graph-Based Fraud Detection")
print("=" * 55)

np.random.seed(42)

N_WORKERS = 50_000
N_DEVICES = 35_000

print("\nBuilding worker-device relationship graph...")
print(f"  Workers : {N_WORKERS:,}")
print(f"  Devices : {N_DEVICES:,}")

# Normal pattern: most workers map to a broad device pool
normal_workers = int(N_WORKERS * 0.97)
fraud_workers = N_WORKERS - normal_workers

normal_worker_ids = np.arange(normal_workers)
normal_device_ids = np.random.randint(0, N_DEVICES - 20, size=normal_workers)

# Fraud pattern: a small group of workers share a very small device pool
fraud_worker_ids = np.arange(normal_workers, N_WORKERS)
fraud_device_ids = np.random.randint(N_DEVICES - 20, N_DEVICES, size=fraud_workers)

edges = []

for worker_id, device_id in zip(normal_worker_ids, normal_device_ids):
    edges.append((f"worker_{worker_id}", f"device_{device_id}"))

for worker_id, device_id in zip(fraud_worker_ids, fraud_device_ids):
    edges.append((f"worker_{worker_id}", f"device_{device_id}"))

print(f"✓ Edge list created: {len(edges):,} edges")

# Build graph and run connected components
t0 = time.time()

G = nx.Graph()
G.add_edges_from(edges)

components = list(nx.connected_components(G))

t1 = time.time()

# Flag unusually large connected components
suspicious_components = [component for component in components if len(component) >= 50]

flagged_workers = []
for component in suspicious_components:
    flagged_workers.extend([
        node for node in component
        if node.startswith("worker_")
    ])

flagged_count = len(flagged_workers)
clean_count = N_WORKERS - flagged_count

print(f"✓ Connected components time: {(t1 - t0) * 1000:.0f} ms")
print(f"✓ Graph nodes: {G.number_of_nodes():,}")
print(f"✓ Graph edges: {G.number_of_edges():,}")

print("\nFraud cluster analysis:")
print(f"  Total components    : {len(components):,}")
print(f"  Suspicious clusters : {len(suspicious_components):,}")
print(f"  Workers flagged     : {flagged_count:,} ({flagged_count / N_WORKERS:.1%})")

print()
print("✓ Fraud gate complete")
print(f"✓ Clean workers proceeding to credit scoring: {clean_count:,}")

print()
print("Implementation note:")
print("  NetworkX is used here for simplicity and Colab stability.")
print("  In production, this graph stage would run on NVIDIA cuGraph")
print("  as part of the GPU pipeline.")

# Store clean worker IDs for next stage
flagged_worker_set = set(flagged_workers)

clean_worker_ids = [
    int(node.replace("worker_", ""))
    for node in G.nodes
    if node.startswith("worker_") and node not in flagged_worker_set
]

Stage 3: Graph-Based Fraud Detection

Building worker-device relationship graph...
  Workers : 50,000
  Devices : 35,000
✓ Edge list created: 50,000 edges
✓ Connected components time: 489 ms
✓ Graph nodes: 76,231
✓ Graph edges: 50,000

Fraud cluster analysis:
  Total components    : 26,231
  Suspicious clusters : 20
  Workers flagged     : 1,500 (3.0%)

✓ Fraud gate complete
✓ Clean workers proceeding to credit scoring: 48,500

Implementation note:
  etworkX is used here for simplicity and Colab stability.
  In production, this graph stage would run on NVIDIA cuGraph
  as part of the GPU pipeline.


Stage 4 – GPU Credit Scoring with XGBoost

Clean workers from the fraud gate proceed to credit scoring.

For gig workers, bureau data is often thin or missing. So this model uses platform and repayment signals instead of depending only on traditional credit history.

The features include:

- earnings velocity
- platform tenure
- payment consistency
- repayment capacity
- current obligation load
- identity risk signals

In this stage, I train an XGBoost model using GPU acceleration. The goal is not to build a final production scorecard, but to show how the risk scoring layer fits after fraud screening in the GPU decisioning pipeline.

In [9]:
import xgboost as xgb
import numpy as np
import pandas as pd
import time
from sklearn.model_selection import train_test_split

print("=" * 55)
print("Stage 4: GPU Credit Scoring — XGBoost")
print("=" * 55)

np.random.seed(42)
N = 50_000

df_credit = pd.DataFrame({
    "monthly_earnings_usd":      np.random.lognormal(mean=7.8, sigma=0.4, size=N).round(2),
    "trips_per_month":           np.random.poisson(lam=180, size=N),
    "platform_tenure_months":    np.random.randint(1, 60, size=N),
    "fulfillment_rating":        np.clip(np.random.normal(4.5, 0.3, N), 1.0, 5.0).round(2),
    "active_days_per_month":     np.random.randint(10, 28, size=N),
    "payment_consistency_score": np.clip(np.random.normal(0.78, 0.12, N), 0, 1).round(3),
    "foir_current":              np.clip(np.random.normal(0.38, 0.12, N), 0.05, 0.85).round(3),
    "existing_loan_count":       np.random.poisson(lam=0.8, size=N),
    "kyc_verification_score":    np.clip(np.random.normal(0.85, 0.1, N), 0, 1).round(3),
})

# Engineer features
df_credit["earnings_per_trip"]      = df_credit["monthly_earnings_usd"] / (df_credit["trips_per_month"] + 1)
df_credit["earnings_stability"]     = (df_credit["payment_consistency_score"] * 0.6 + (df_credit["active_days_per_month"] / 28.0) * 0.4)
df_credit["tenure_composite"]       = ((df_credit["platform_tenure_months"] / 60.0) * 0.5 + (df_credit["fulfillment_rating"] / 5.0) * 0.5)
df_credit["repayment_capacity"]     = (1.0 - df_credit["foir_current"]) * df_credit["earnings_stability"]
df_credit["available_income_ratio"] = 1.0 - df_credit["foir_current"]

# Build correlated target — repayment driven by actual features
repayment_score = (
    df_credit["repayment_capacity"] * 0.4 +
    df_credit["tenure_composite"] * 0.3 +
    df_credit["payment_consistency_score"] * 0.2 +
    df_credit["kyc_verification_score"] * 0.1
)
repayment_score = (repayment_score - repayment_score.min()) / (repayment_score.max() - repayment_score.min())
noise = np.random.normal(0, 0.1, N)
df_credit["repaid_on_time"] = ((repayment_score + noise) > 0.5).astype(int)

features = [
    "earnings_per_trip", "earnings_stability", "tenure_composite",
    "repayment_capacity", "available_income_ratio",
    "payment_consistency_score", "platform_tenure_months",
    "fulfillment_rating", "foir_current", "existing_loan_count"
]

X = df_credit[features]
y = df_credit["repaid_on_time"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"\nTraining XGBoost on GPU...")
print(f"  Training records : {len(X_train):,}")
print(f"  Test records     : {len(X_test):,}")

dtrain = xgb.DMatrix(X_train, label=y_train)
dtest  = xgb.DMatrix(X_test,  label=y_test)

params = {
    "device":        "cuda",
    "tree_method":   "hist",
    "objective":     "binary:logistic",
    "eval_metric":   "auc",
    "max_depth":     6,
    "learning_rate": 0.1,
    "subsample":     0.8,
    "random_state":  42,
}

t0 = time.time()
model = xgb.train(
    params, dtrain,
    num_boost_round=200,
    evals=[(dtest, "test")],
    verbose_eval=50
)
t1 = time.time()

print(f"\n✓ GPU XGBoost training completed: {(t1-t0)*1000:.0f}ms")
print("✓ In production, this stage benefits more when training data already stays on GPU")

# Score and credit decisions
proba = model.predict(dtest)

MAX_LIMIT = 2000
df_decisions = X_test.copy()
df_decisions["repayment_probability"] = proba
df_decisions["credit_decision"]       = np.where(proba >= 0.65, "APPROVE", "DECLINE")
df_decisions["credit_limit_usd"]      = np.where(
    proba >= 0.65,
    (proba * MAX_LIMIT * df_decisions["repayment_capacity"]).round(0),
    0
)

approved  = (df_decisions["credit_decision"] == "APPROVE").sum()
declined  = (df_decisions["credit_decision"] == "DECLINE").sum()
avg_limit = df_decisions[df_decisions["credit_limit_usd"] > 0]["credit_limit_usd"].mean()

print(f"\nCredit decision summary:")
print(f"  Approved  : {approved:,} ({approved/len(df_decisions):.1%})")
print(f"  Declined  : {declined:,} ({declined/len(df_decisions):.1%})")
print(f"  Avg limit : ${avg_limit:,.0f}")
print("\n✓ Credit scoring complete")
print("✓ Dynamic limits tied to repayment capacity, not static origination")
print("✓ GPU XGBoost training completed")

Stage 4: GPU Credit Scoring — XGBoost

Training XGBoost on GPU...
  Training records : 40,000
  Test records     : 10,000
[0]	test-auc:0.85634
[50]	test-auc:0.86940
[100]	test-auc:0.86824
[150]	test-auc:0.86738
[199]	test-auc:0.86639

✓ GPU XGBoost training completed: 437ms
✓ In production, this stage benefits more when training data already stays on GPU

Credit decision summary:
  Approved  : 2,986 (29.9%)
  Declined  : 7,014 (70.1%)
  Avg limit : $945

✓ Credit scoring complete
✓ Dynamic limits tied to repayment capacity, not static origination
✓ GPU XGBoost training completed


Stage 5 – Explainability and Audit Trail

Every credit decision needs an explanation.

For approvals, the explanation helps understand why the worker qualified and what limit was assigned.

For declines, the explanation is more important. The system needs to produce a clear reason that a risk, compliance, or operations team can review.

In this stage, I use SHAP to generate feature attribution for the decision. I also create a simple audit record with the model version, decision, probability score, limit, top factors, and timestamp.

The goal is to show that explainability and audit logging are part of the inference flow, not an afterthought.